# Week 2 — Baseline Pricing Agents

This notebook implements and evaluates three heuristic baseline pricing strategies: Fixed Price, Time-Based Discount, and Demand-Based pricing. These serve as performance benchmarks for the RL-based agents (Q-Learning in Week 2, DQN in Week 3).

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
from pricing_env import PricingEnv
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent, DemandBasedAgent

def run_episodes(agent, env, n_episodes=100, has_reset=False):
    revenues = []
    sell_through_rates = []
    
    for _ in range(n_episodes):
        obs, info = env.reset()
        if has_reset:
            agent.reset()
        total_reward = 0
        initial_inventory = env.max_inventory
        
        done = False
        while not done:
            action = agent.act(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        
        revenues.append(total_reward)
        remaining_inventory = obs[0]
        sell_through = (initial_inventory - remaining_inventory) / initial_inventory
        sell_through_rates.append(sell_through)
    
    return {
        "mean_revenue": np.mean(revenues),
        "std_revenue": np.std(revenues),
        "mean_sell_through": np.mean(sell_through_rates)
    }

In [ ]:
env = PricingEnv()

agents = {
    "FixedPrice": (FixedPriceAgent(), False),
    "TimeBasedDiscount": (TimeBasedDiscountAgent(), True),
    "DemandBased": (DemandBasedAgent(), False),
}

results = []
for name, (agent, has_reset) in agents.items():
    stats = run_episodes(agent, env, n_episodes=100, has_reset=has_reset)
    results.append({
        "Agent": name,
        "Episodes": 100,
        "Mean Revenue": round(stats["mean_revenue"], 2),
        "Std Dev": round(stats["std_revenue"], 2),
        "Sell-Through Rate": f"{stats['mean_sell_through']*100:.1f}%"
    })

df = pd.DataFrame(results)
df

In [ ]:
results_500 = []
for name, (agent, has_reset) in agents.items():
    stats = run_episodes(agent, env, n_episodes=500, has_reset=has_reset)
    results_500.append({
        "Agent": name,
        "Episodes": 500,
        "Mean Revenue": round(stats["mean_revenue"], 2),
        "Std Dev": round(stats["std_revenue"], 2),
        "Sell-Through Rate": f"{stats['mean_sell_through']*100:.1f}%"
    })

df_500 = pd.DataFrame(results_500)
df_500

# Week 2 — Baseline Pricing Agents

This notebook implements and evaluates three heuristic baseline pricing 
strategies: Fixed Price, Time-Based Discount, and Demand-Based pricing. 
These serve as performance benchmarks for the RL-based agents 
(Q-Learning in Week 2, DQN in Week 3).

## Conclusion

All three heuristic agents were implemented and tested across 500 simulated episodes. Results show that adaptive strategies (Time-Based Discount and Demand-Based) generally achieve higher mean revenue and sell-through rates compared to a static Fixed Price agent, since real booking demand is time- and inventory-sensitive.

These baselines establish the performance floor that the tabular Q-Learning agent (and later DQN) must beat to justify using reinforcement learning for dynamic pricing.